# NLP Lab03 - Syntax Trees and Parsing

---

| Part | Topic | Lib |
|---|---|---|
| 1 | Constituency trees, CFG | `nltk.ChartParser` |
| 2 | Chomsky Normal Form, CYK | `nltk.ChartParser` |
| 3 | PCFG, probabilistic CYK | `nltk.ViterbiParser` |
| 4 | Dependency trees | `spaCy` |


In [ ]:
# Uncomment and run once if these are not installed yet
# !pip install -q nltk spacy
# !python -m spacy download en_core_web_sm

In [2]:
import random
from collections import defaultdict, Counter

import nltk
from nltk import CFG, PCFG, Nonterminal, Tree, induce_pcfg
from nltk.grammar import Production
from nltk.parse import ChartParser, ViterbiParser

import spacy
from spacy import displacy

random.seed(13)
nlp = spacy.load("en_core_web_sm")

print("nltk  :", nltk.__version__)
print("spaCy :", spacy.__version__)


def check(label, got, want):
    """Compare without raising, so an unfinished TODO still runs."""
    ok = got == want
    print(("PASS" if ok else "FAIL"), "|", label)
    if not ok:
        print("       got     :", got)
        print("       expected:", want)
    return ok

nltk  : 3.10.3
spaCy : 3.8.16


---

# Part 1 - Constituency Trees and CFG

A context-free grammar is a set of rewrite rules. Every rule has the form
`A -> anything`, where the left side is always a single nonterminal.

A constituency tree is a record of which rules were applied to produce a
sentence. One sentence can have more than one such record. That is
**structural ambiguity**.

## 1.1 A sample grammar

Watch these two rules. They are the source of nearly all the ambiguity in
this notebook.

```
NP -> NP PP     the PP attaches to a noun  - "the man who has a telescope"
VP -> VP PP     the PP attaches to a verb  - "saw, by means of a telescope"
```

In [3]:
grammar = CFG.fromstring("""
    S   -> NP VP
    NP  -> Det N | NP PP | 'I'
    VP  -> V NP | VP PP
    PP  -> P NP
    Det -> 'a' | 'the'
    N   -> 'man' | 'telescope' | 'park' | 'boy' | 'dog' | 'hill'
    V   -> 'saw'
    P   -> 'with' | 'in' | 'on'
""")

SENT1 = "I saw a man with a telescope".split()

parser = ChartParser(grammar)
trees = list(parser.parse(SENT1))

print("number of possible trees:", len(trees))
print()
for i, t in enumerate(trees, 1):
    print(f"--- tree {i} ---")
    t.pretty_print()

number of possible trees: 2

--- tree 1 ---
     S                                    
  ___|___________                          
 |               VP                       
 |        _______|________                 
 |       VP               PP              
 |    ___|___         ____|___             
 |   |       NP      |        NP          
 |   |    ___|___    |     ___|______      
 NP  V  Det      N   P   Det         N    
 |   |   |       |   |    |          |     
 I  saw  a      man with  a      telescope

--- tree 2 ---
     S                                    
  ___|___________                          
 |               VP                       
 |    ___________|___                      
 |   |               NP                   
 |   |        _______|____                 
 |   |       |            PP              
 |   |       |        ____|___             
 |   |       NP      |        NP          
 |   |    ___|___    |     ___|______      
 NP  V  Det      N   P   De

Read both trees above and make sure you can say which one means
*"I used a telescope to see him"* and which one means
*"the man was carrying a telescope"*.

An NLTK `Tree` behaves like a nested list, so you can index into it directly.

In [4]:
t = trees[0]
print(t)
print("root label     :", t.label())
print("children       :", [child.label() for child in t])
print("leaves         :", t.leaves())
print("height         :", t.height())
print("productions used:")
for p in t.productions()[:6]:
    print("   ", p)
print()
print("bracketed form:", t)

(S
  (NP I)
  (VP
    (VP (V saw) (NP (Det a) (N man)))
    (PP (P with) (NP (Det a) (N telescope)))))
root label     : S
children       : ['NP', 'VP']
leaves         : ['I', 'saw', 'a', 'man', 'with', 'a', 'telescope']
height         : 6
productions used:
    S -> NP VP
    NP -> 'I'
    VP -> VP PP
    VP -> V NP
    V -> 'saw'
    NP -> Det N

bracketed form: (S
  (NP I)
  (VP
    (VP (V saw) (NP (Det a) (N man)))
    (PP (P with) (NP (Det a) (N telescope)))))


## TODO 1 - Extend the grammar with coordination

Build `grammar_1` by taking `grammar` and adding:

1. the verb `watched`
2. the word `and` as a category `Conj`
3. rules that coordinate two NPs, and rules that coordinate two VPs

The test sentence is `the boy watched a dog and a man`, and it should parse in
**exactly 1 way**.

In [22]:
# TODO 1 - write your grammar here
## HINT: which word are not in grammar
grammar_1 = CFG.fromstring("""
    S    -> NP VP
        NP   -> Det N | NP PP | NP Conj NP | 'I'
        VP   -> V NP | VP PP
        PP   -> P NP
        Det  -> 'a' | 'the'
        N    -> 'man' | 'park' | 'boy' | 'hill' | 'dog'
        V    -> 'saw' | 'watched'
        P    -> 'with' | 'in' | 'on'
        Conj -> 'and'
""")
###
SENT_CONJ = "the boy watched a dog and a man".split()


try:
    n_parses = len(list(ChartParser(grammar_1).parse(SENT_CONJ)))
except ValueError as e:
    n_parses = f"ERROR: grammar does not cover some word: {e}"

check("grammar_1 parses 'the boy watched a dog and a man' in 1 way", n_parses, 1)

PASS | grammar_1 parses 'the boy watched a dog and a man' in 1 way


True

---

# Part 2 - Chomsky Normal Form and CYK

CYK needs a grammar in which every rule has one of these two shapes:

```
A -> B C        exactly two nonterminals
A -> 'word'     exactly one terminal
```

That is **Chomsky Normal Form**. Every CFG can be converted to CNF without
changing the language it generates.

## 2.1 Convert a tree to CNF and back

NLTK converts at the *tree* level. A rule with more than two children gets
split using temporary nodes whose labels contain a `|`.

In [6]:
demo = Tree.fromstring(
    "(S (NP (Det the) (N boy)) (VP (V gave) (NP (Det a) (N dog)) (PP (P to) (NP (N me)))))"
)
print("before - VP has three children")
demo.pretty_print()

before - VP has three children
                  S                     
      ____________|_______               
     |                    VP            
     |        ____________|_______       
     |       |        |           PP    
     |       |        |        ___|___   
     NP      |        NP      |       NP
  ___|___    |     ___|___    |       |  
Det      N   V   Det      N   P       N 
 |       |   |    |       |   |       |  
the     boy gave  a      dog  to      me



In [7]:
cnf = demo.copy(deep=True)
cnf.chomsky_normal_form()
print("after chomsky_normal_form()")
cnf.pretty_print()

after chomsky_normal_form()
                  S                                
      ____________|_______                          
     |                    VP                       
     |        ____________|______                   
     |       |               VP|<NP-PP>            
     |       |         __________|___________       
     |       |        |                      PP    
     |       |        |                   ___|___   
     NP      |        NP                 |       NP
  ___|___    |     ___|___               |       |  
Det      N   V   Det      N              P       N 
 |       |   |    |       |              |       |  
the     boy gave  a      dog             to      me



In [8]:
back = cnf.copy(deep=True)
back.un_chomsky_normal_form()
print("back", back)
print("round trip recovers the original:", back == demo)

back (S
  (NP (Det the) (N boy))
  (VP (V gave) (NP (Det a) (N dog)) (PP (P to) (NP (N me)))))
round trip recovers the original: True


## 2.2 The CNF grammar we will run CYK on

This grammar is written in CNF from the start, so no conversion is needed.

In [9]:
G_CNF = CFG.fromstring("""
    S   -> NP VP
    NP  -> Det N | NP PP
    NP  -> 'I'
    VP  -> V NP | VP PP
    PP  -> P NP
    Det -> 'a' | 'the'
    N   -> 'man' | 'telescope' | 'park'
    V   -> 'saw'
    P   -> 'with' | 'in'
""")

print("already in CNF:", G_CNF.is_chomsky_normal_form())
print("start symbol  :", G_CNF.start())
print()
print("rules whose right side starts with 'saw':", G_CNF.productions(rhs="saw"))

already in CNF: True
start symbol  : S

rules whose right side starts with 'saw': [V -> 'saw']


## Write CYK with backpointers

A recognizer only answers yes or no. To get trees out, each nonterminal in
each cell has to remember which rule produced it and where the span was split.

Change every cell from a `set` into a `dict` mapping each nonterminal to a
**list of sources**. One source is `(k, B, C)` for a binary rule, or the word
itself for a lexical rule.

Write `cyk_parse(g, words)` returning **every** possible tree as an
`nltk.Tree`. The count has to match `ChartParser`.

Two separate stages: fill the table, then walk the backpointers to assemble
trees.

`build` is a cross product. If the left child can be built 3 ways and the right
child 2 ways, that rule yields 6 trees. This is exactly the exponential growth
you counted in part 1.

One `defaultdict` trap: `B in table[i][k]` is safe because membership tests do
not create keys, but writing `table[i][k][B]` does create one.

In [10]:
def cyk_parse(g, words):
    n = len(words)
    table = [[defaultdict(list) for _ in range(n + 1)] for _ in range(n + 1)]

    for j in range(1, n + 1):
        for prod in g.productions(rhs=words[j - 1]):
            if len(prod.rhs()) == 1:
                table[j - 1][j][prod.lhs()].append(words[j - 1])

        for i in range(j - 2, -1, -1):
            for k in range(i + 1, j):
                for prod in g.productions():
                    if len(prod.rhs()) != 2:
                        continue
                    B, C = prod.rhs()
                    if B in table[i][k] and C in table[k][j]:
                        table[i][j][prod.lhs()].append((k, B, C))

    def build(i, j, A):
        out = []
        for bp in table[i][j][A]:
            if isinstance(bp, str):
                out.append(Tree(A.symbol(), [bp]))
            else:
                k, B, C = bp
                for left in build(i, k, B):
                    for right in build(k, j, C):
                        out.append(Tree(A.symbol(), [left, right]))
        return out

    return build(0, n, g.start())


my_trees = cyk_parse(G_CNF, SENT1)
ref_trees = list(ChartParser(G_CNF).parse(SENT1))

check("CYK tree count matches ChartParser", len(my_trees), len(ref_trees))
check("the two sets of trees are identical",
      sorted(str(t) for t in my_trees), sorted(str(t) for t in ref_trees))

for t in my_trees:
    t.pretty_print()

PASS | CYK tree count matches ChartParser
PASS | the two sets of trees are identical
     S                                    
  ___|___________                          
 |               VP                       
 |    ___________|___                      
 |   |               NP                   
 |   |        _______|____                 
 |   |       |            PP              
 |   |       |        ____|___             
 |   |       NP      |        NP          
 |   |    ___|___    |     ___|______      
 NP  V  Det      N   P   Det         N    
 |   |   |       |   |    |          |     
 I  saw  a      man with  a      telescope

     S                                    
  ___|___________                          
 |               VP                       
 |        _______|________                 
 |       VP               PP              
 |    ___|___         ____|___             
 |   |       NP      |        NP          
 |   |    ___|___    |     ___|______      
 

---

# Part 3 - PCFG and Probabilistic CYK

A CFG says what is *grammatical*. It says nothing about what is *likely*.
A PCFG attaches a probability to every rule, and the rules sharing a left side
must sum to 1.

The probability of a tree is the product over the rules it uses, so parsing
becomes a search for the highest-scoring product. That is Viterbi running over
the CYK table.

In [ ]:
PG = PCFG.fromstring("""
    S   -> NP VP        [1.0]
    NP  -> Det N        [0.4]
    NP  -> NP PP        [0.2]
    NP  -> 'I'          [0.4]
    VP  -> V NP         [0.6]
    VP  -> VP PP        [0.4]
    PP  -> P NP         [1.0]
    Det -> 'a'          [0.5]
    Det -> 'the'        [0.5]
    N   -> 'man'        [0.4]
    N   -> 'telescope'  [0.3]
    N   -> 'park'       [0.3]
    V   -> 'saw'        [1.0]
    P   -> 'with'       [0.6]
    P   -> 'in'         [0.4]
""")

print("in CNF:", PG.is_chomsky_normal_form())
for lhs in [Nonterminal("NP"), Nonterminal("VP")]:
    ps = PG.productions(lhs=lhs)
    print(f"{lhs} probabilities sum to", round(sum(p.prob() for p in ps), 6))

in CNF: True
NP probabilities sum to 1.0
VP probabilities sum to 1.0


: 

In [12]:
vp_parser = ViterbiParser(PG)
best_ref = vp_parser.parse_one(SENT1)

print("most probable tree according to ViterbiParser")
best_ref.pretty_print()
print("P =", best_ref.prob())

most probable tree according to ViterbiParser
     S                                    
  ___|___________                          
 |               VP                       
 |        _______|________                 
 |       VP               PP              
 |    ___|___         ____|___             
 |   |       NP      |        NP          
 |   |    ___|___    |     ___|______      
 NP  V  Det      N   P   Det         N    
 |   |   |       |   |    |          |     
 I  saw  a      man with  a      telescope

P = 0.00027648000000000006


## Write probabilistic CYK yourself

each cell keeps only the **best** entry per
nonterminal.

```
best[i][j][A]  highest probability of any tree rooted at A covering words[i:j]
back[i][j][A]  where that value came from
```

For a rule `A -> B C` at split point `k`:

```
p = prob(A -> B C) * best[i][k][B] * best[k][j][C]
```

Keep it if it beats what is already stored.

Write `pcyk(pg, words)` returning `(tree, prob)` and compare with
`ViterbiParser`. The numbers must agree to the last digit.

*Hint.* `prod.prob()` gives a rule's probability.

Both routes give 0.00027648. What prints has a floating point tail of
`...00000006`, but the two values are bit-for-bit identical, so `==` passes.

Work it through by hand. The two competing trees are:

```
VP attachment:  P = 0.4 x 1.0 x 0.6 x 0.4 x 0.6 x 0.4 x 0.5 x 0.4 x 1.0 x 0.6 x 0.4 x 0.5 x 0.3
NP attachment:  identical except it uses NP -> NP PP [0.2] instead of VP -> VP PP [0.4]
```

The ratio is exactly 0.4 to 0.2, so VP attachment wins by a factor of two.
That happens because we chose those numbers, not because English works that
way. A real PCFG gets its numbers by counting a treebank

In [13]:
def pcyk(pg, words):
    n = len(words)
    best = [[dict() for _ in range(n + 1)] for _ in range(n + 1)]
    back = [[dict() for _ in range(n + 1)] for _ in range(n + 1)]

    for j in range(1, n + 1):
        for prod in pg.productions(rhs=words[j - 1]):
            if len(prod.rhs()) == 1:
                A, p = prod.lhs(), prod.prob()
                if p > best[j - 1][j].get(A, 0.0):
                    best[j - 1][j][A] = p
                    back[j - 1][j][A] = words[j - 1]

        for i in range(j - 2, -1, -1):
            for k in range(i + 1, j):
                for prod in pg.productions():
                    if len(prod.rhs()) != 2:
                        continue
                    B, C = prod.rhs()
                    if B not in best[i][k] or C not in best[k][j]:
                        continue
                    A = prod.lhs()
                    p = prod.prob() * best[i][k][B] * best[k][j][C]
                    if p > best[i][j].get(A, 0.0):
                        best[i][j][A] = p
                        back[i][j][A] = (k, B, C)

    start = pg.start()
    if start not in best[0][n]:
        return None, 0.0

    def build(i, j, A):
        bp = back[i][j][A]
        if isinstance(bp, str):
            return Tree(A.symbol(), [bp])
        k, B, C = bp
        return Tree(A.symbol(), [build(i, k, B), build(k, j, C)])

    return build(0, n, start), best[0][n][start]


my_tree, my_prob = pcyk(PG, SENT1)
my_tree.pretty_print()
print("P from our pcyk     :", my_prob)
print("P from ViterbiParser:", best_ref.prob())
print()
check("probabilities agree", my_prob, best_ref.prob())
check("tree structures agree", str(my_tree), str(Tree.convert(best_ref)))

     S                                    
  ___|___________                          
 |               VP                       
 |        _______|________                 
 |       VP               PP              
 |    ___|___         ____|___             
 |   |       NP      |        NP          
 |   |    ___|___    |     ___|______      
 NP  V  Det      N   P   Det         N    
 |   |   |       |   |    |          |     
 I  saw  a      man with  a      telescope

P from our pcyk     : 0.00027648000000000006
P from ViterbiParser: 0.00027648000000000006

PASS | probabilities agree
PASS | tree structures agree


True

## Read a PCFG off a treebank

The probabilities above were invented. Real ones come from counting trees.

$$P(A \rightarrow \beta) = \frac{\text{count}(A \rightarrow \beta)}{\text{count}(A)}$$

Write `induce_manual(trees, start)` returning a `dict` from `Production` to
`float`, and compare it against `nltk.induce_pcfg`.

*Hint.* `tree.productions()` gives every rule that tree used.

In [14]:
MINI_TREEBANK = [
    "(S (NP (Det the) (N boy)) (VP (V saw) (NP (Det a) (N dog))))",
    "(S (NP (Det a) (N man)) (VP (V saw) (NP (Det the) (N park))))",
    "(S (NP (Det the) (N dog)) (VP (V saw) (NP (Det the) (N boy))))",
    "(S (NP (Det the) (N boy)) (VP (VP (V saw) (NP (Det a) (N man))) (PP (P in) (NP (Det the) (N park)))))",
    "(S (NP (Det a) (N dog)) (VP (VP (V saw) (NP (Det a) (N man))) (PP (P with) (NP (Det a) (N telescope)))))",
    "(S (NP (NP (Det the) (N man)) (PP (P with) (NP (Det a) (N telescope)))) (VP (V saw) (NP (Det the) (N dog))))",
]

tb = [Tree.fromstring(s) for s in MINI_TREEBANK]
print("trees in the treebank:", len(tb))
tb[3].pretty_print()

trees in the treebank: 6
                 S                          
      ___________|_______                    
     |                   VP                 
     |            _______|_______            
     |           VP              PP         
     |        ___|___         ___|___        
     NP      |       NP      |       NP     
  ___|___    |    ___|___    |    ___|___    
Det      N   V  Det      N   P  Det      N  
 |       |   |   |       |   |   |       |   
the     boy saw  a      man  in the     park



In [15]:
ref_pcfg = induce_pcfg(Nonterminal("S"), [p for t in tb for p in t.productions()])
ref_map = {Production(p.lhs(), p.rhs()): p.prob() for p in ref_pcfg.productions()}

Two counters: one for whole rules, one for left sides alone. Then divide.

Notice that this tiny treebank gives `VP -> V NP` 0.667 and `VP -> VP PP`
0.333, which is the opposite ordering from the PCFG we hand-wrote a moment
ago. That flips which reading the parser prefers. The numbers in a PCFG are
entirely a property of the corpus it was counted from.

In [16]:
def induce_manual(trees, start):
    rule_count = Counter()
    lhs_count = Counter()
    for t in trees:
        for p in t.productions():
            rule_count[p] += 1
            lhs_count[p.lhs()] += 1
    return {p: c / lhs_count[p.lhs()] for p, c in rule_count.items()}


manual = induce_manual(tb, Nonterminal("S"))

print(f"{'rule':<28}{'by hand':>10}{'induce_pcfg':>14}")
print("-" * 52)
for p in sorted(manual, key=lambda x: (str(x.lhs()), str(x.rhs()))):
    print(f"{str(p):<28}{manual[p]:>10.4f}{ref_map[p]:>14.4f}")

same = all(abs(manual[k] - v) < 1e-12 for k, v in ref_map.items()) and len(manual) == len(ref_map)
print()
check("hand-counted probabilities match induce_pcfg", same, True)

rule                           by hand   induce_pcfg
----------------------------------------------------
Det -> 'a'                      0.4667        0.4667
Det -> 'the'                    0.5333        0.5333
N -> 'boy'                      0.2000        0.2000
N -> 'dog'                      0.2667        0.2667
N -> 'man'                      0.2667        0.2667
N -> 'park'                     0.1333        0.1333
N -> 'telescope'                0.1333        0.1333
NP -> Det N                     0.9375        0.9375
NP -> NP PP                     0.0625        0.0625
P -> 'in'                       0.3333        0.3333
P -> 'with'                     0.6667        0.6667
PP -> P NP                      1.0000        1.0000
S -> NP VP                      1.0000        1.0000
V -> 'saw'                      1.0000        1.0000
VP -> V NP                      0.7500        0.7500
VP -> VP PP                     0.2500        0.2500

PASS | hand-counted probabilities match induc

True

---

# Part 4 - Dependency Trees

A constituency tree answers *which words group together*.
A dependency tree answers *which word depends on which*.

A dependency tree is a graph where every word has exactly one head, there is
exactly one root, and there are no cycles. The whole structure therefore fits
in a single list: `heads[i]` is the index of the head of word `i`.

In [17]:
doc = nlp("I saw a man with a telescope in the park .")

print(f"{'i':>3} {'word':<12}{'POS':<7}{'dep':<10}{'head':<12}")
print("-" * 48)
for tok in doc:
    head = "ROOT" if tok.head.i == tok.i else f"{tok.head.text} ({tok.head.i})"
    print(f"{tok.i:>3} {tok.text:<12}{tok.pos_:<7}{tok.dep_:<10}{head:<12}")

  i word        POS    dep       head        
------------------------------------------------
  0 I           PRON   nsubj     saw (1)     
  1 saw         VERB   ROOT      ROOT        
  2 a           DET    det       man (3)     
  3 man         NOUN   dobj      saw (1)     
  4 with        ADP    prep      man (3)     
  5 a           DET    det       telescope (6)
  6 telescope   NOUN   pobj      with (4)    
  7 in          ADP    prep      saw (1)     
  8 the         DET    det       park (9)    
  9 park        NOUN   pobj      in (7)      
 10 .           PUNCT  punct     saw (1)     


In [18]:
displacy.render(doc, style="dep", jupyter=True,
                options={"compact": True, "distance": 95})